# BRFSS 2015 · MLP 5 thuộc tính cho web

Dùng bản CSV nhị phân đầy đủ. Năm thuộc tính là sức khỏe tự đánh giá, BMI, tiền sử huyết áp cao, nhóm tuổi và tiền sử cholesterol cao. Đây là nhóm được chọn sau khi so sánh các tập biến trên BRFSS 2015.

Chia train/validation/test theo tỷ lệ 70/15/15 có phân tầng. Fit bộ chuẩn hóa trên train gốc. Chỉ lấy mẫu lại lớp 1 trong train với mục tiêu khoảng 1 mẫu lớp 1 trên 2 mẫu lớp 0. Validation và test giữ phân bố gốc.

Huấn luyện một MLP tối đa 25 epoch, dừng sớm theo Average Precision (AP) trên validation. Chọn ngưỡng tối đa F1 lớp 1 trên validation. Báo cáo test ở ngưỡng 0,50 và ngưỡng đã chọn. Các đường accuracy theo epoch dùng cố định ngưỡng 0,50 để so sánh nhất quán.

In [ ]:
from copy import deepcopy
from pathlib import Path
from time import perf_counter

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    ConfusionMatrixDisplay, PrecisionRecallDisplay, RocCurveDisplay,
    accuracy_score, average_precision_score, classification_report, log_loss,
    precision_recall_curve, roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits

SEED = 42
MAX_EPOCHS = 25
PATIENCE = 5
MIN_DELTA = 0.0005
POS_TO_NEG = 0.5
FEATURES = ["GenHlth", "BMI", "HighBP", "Age", "HighChol"]
TARGET = "Diabetes_binary"
DATA_ROOT = Path("/kaggle/input/notebooks/alexteboul/diabetes-health-indicators-dataset-notebook")
DATA_FILE = "diabetes_binary_health_indicators_BRFSS2015.csv"
OUTPUT_DIR = Path("/kaggle/working")
IMPORTANCE_REPEATS = 10

def finish_plot(fig, filename):
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / filename, dpi=170, bbox_inches="tight")
    plt.show()
    plt.close(fig)

In [ ]:
paths = sorted(DATA_ROOT.rglob(DATA_FILE))
if len(paths) != 1:
    raise RuntimeError(f"Expected one {DATA_FILE} under {DATA_ROOT}, found {paths}")

df = pd.read_csv(paths[0], usecols=[*FEATURES, TARGET])
if not np.isfinite(df.to_numpy(dtype=float)).all():
    raise ValueError("Unexpected missing or non-finite values.")
if set(df[TARGET].unique()) != {0, 1}:
    raise ValueError("Expected target labels 0 and 1.")

y = df[TARGET].astype(int).to_numpy()
train, holdout = train_test_split(np.arange(len(df)), test_size=0.30, stratify=y, random_state=SEED)
val, test = train_test_split(holdout, test_size=0.50, stratify=y[holdout], random_state=SEED)

x = df[FEATURES].astype(np.float32)
scaler = StandardScaler().fit(x.iloc[train])
x_train = scaler.transform(x.iloc[train])
x_val = scaler.transform(x.iloc[val])
x_test = scaler.transform(x.iloc[test])
y_train, y_val, y_test = y[train], y[val], y[test]

rng = np.random.default_rng(SEED)
positive = np.flatnonzero(y_train == 1)
negative_count = int((y_train == 0).sum())
extra_count = max(0, round(POS_TO_NEG * negative_count) - len(positive))
sampled = np.concatenate([np.arange(len(train)), rng.choice(positive, size=extra_count, replace=True)])
rng.shuffle(sampled)
fit_x, fit_y = x_train[sampled], y_train[sampled]
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

splits = {"Train gốc": y_train, "Train lấy mẫu lại": fit_y, "Validation": y_val, "Test": y_test}
counts = pd.DataFrame({label: np.bincount(values, minlength=2) for label, values in splits.items()},
                      index=["Lớp 0", "Lớp 1"]).T
display(counts.assign(ty_le_lop_1=counts["Lớp 1"] / counts.sum(axis=1)))
print(f"Dữ liệu: {paths[0]} | Dòng train được lấy mẫu thêm: {extra_count:,}")

fig, ax = plt.subplots(figsize=(8, 4))
counts.plot.bar(ax=ax, rot=0, title="Phân bố nhãn trước và sau xử lý lệch lớp")
ax.set_ylabel("Số dòng")
finish_plot(fig, "01_class_balance.png")

fig, axes = plt.subplots(2, 3, figsize=(12, 6))
for feature, ax in zip(FEATURES, axes.flat):
    bins = np.histogram_bin_edges(x.iloc[train][feature], bins=20) if feature == "BMI" else np.arange(-0.5, x[feature].max() + 1.5)
    for label, color in [(0, "tab:blue"), (1, "tab:orange")]:
        ax.hist(x.iloc[train].loc[y_train == label, feature],
                bins=bins,
                density=True, histtype="step", linewidth=1.5, color=color, label=f"Lớp {label}")
    ax.set_title(feature)
axes.flat[0].legend()
axes.flat[-1].set_axis_off()
fig.suptitle("Phân bố đặc trưng theo lớp trên tập train gốc")
finish_plot(fig, "02_train_feature_distributions.png")

In [ ]:
model = MLPClassifier(
    hidden_layer_sizes=(64, 32), activation="relu", solver="adam",
    batch_size=1024, alpha=0.001, random_state=SEED,
)
history = []
best_ap = -np.inf
patience_anchor = -np.inf
stale = 0
started = perf_counter()

with threadpool_limits(limits=2, user_api="blas"):
    for epoch in range(1, MAX_EPOCHS + 1):
        model.partial_fit(fit_x, fit_y, classes=[0, 1])
        train_score = model.predict_proba(x_train)[:, 1]
        val_score = model.predict_proba(x_val)[:, 1]
        val_ap = average_precision_score(y_val, val_score)
        history.append({
            "epoch": epoch,
            "train_loss": log_loss(y_train, train_score),
            "val_loss": log_loss(y_val, val_score),
            "train_acc": accuracy_score(y_train, train_score >= 0.5),
            "val_acc": accuracy_score(y_val, val_score >= 0.5),
            "train_ap": average_precision_score(y_train, train_score),
            "val_ap": val_ap,
        })
        print(f"Epoch {epoch:02d}/{MAX_EPOCHS} | val AP {val_ap:.4f} | {perf_counter()-started:.1f}s", flush=True)
        if val_ap > best_ap:
            best_ap, best_epoch = val_ap, epoch
            best_model = deepcopy(model)
            best_val_score = val_score
        if val_ap > patience_anchor + MIN_DELTA:
            patience_anchor = val_ap
            stale = 0
        else:
            stale += 1
            if stale >= PATIENCE:
                print(f"Early stop: {PATIENCE} epoch không cải thiện đủ.")
                break

curves = pd.DataFrame(history).set_index("epoch")
curves.to_csv(OUTPUT_DIR / "brfss_learning_history.csv")
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
curves[["train_loss", "val_loss"]].plot(ax=axes[0], title="Log loss theo epoch (phân bố gốc)")
curves[["train_acc", "val_acc"]].plot(ax=axes[1], title="Accuracy theo epoch (ngưỡng 0,50)")
axes[1].axhline(np.mean(y_val == 0), color="tab:orange", linestyle=":", label="Baseline luôn đoán lớp 0")
curves[["train_ap", "val_ap"]].plot(ax=axes[2], title="Average Precision theo epoch")
for ax in axes:
    ax.axvline(best_epoch, color="gray", linestyle="--", label="Epoch được chọn")
    ax.set_xlabel("Epoch")
    ax.legend()
finish_plot(fig, "03_learning_curves.png")

In [ ]:
precision, recall, thresholds = precision_recall_curve(y_val, best_val_score)
total = precision[:-1] + recall[:-1]
f1 = np.divide(2 * precision[:-1] * recall[:-1], total, out=np.zeros_like(total), where=total > 0)
threshold = float(thresholds[np.argmax(f1)])
print(f"Epoch được chọn: {best_epoch} | Ngưỡng tối đa F1 trên validation: {threshold:.4f}")

fig, ax = plt.subplots(figsize=(8, 4))
for values, label in [(precision[:-1], "Precision"), (recall[:-1], "Recall"), (f1, "F1")]:
    ax.plot(thresholds, values, label=label)
ax.axvline(threshold, color="gray", linestyle="--", label="Ngưỡng được chọn")
ax.set(xlabel="Ngưỡng", ylabel="Chỉ số lớp 1", title="Chọn ngưỡng trên validation")
ax.legend()
finish_plot(fig, "04_validation_threshold.png")

test_score = best_model.predict_proba(x_test)[:, 1]
ap = float(average_precision_score(y_test, test_score))
auc = float(roc_auc_score(y_test, test_score))
print(f"Test ROC-AUC: {auc:.4f} | AP: {ap:.4f} | AP baseline: {y_test.mean():.4f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
RocCurveDisplay.from_predictions(y_test, test_score, ax=axes[0], name="MLP")
axes[0].plot([0, 1], [0, 1], linestyle="--", color="gray")
PrecisionRecallDisplay.from_predictions(y_test, test_score, ax=axes[1], name="MLP")
axes[1].axhline(y_test.mean(), linestyle="--", color="gray", label="Tỷ lệ lớp 1")
axes[1].legend()
finish_plot(fig, "05_test_roc_pr.png")

reports = {}
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
for col, (name, cutoff) in enumerate([("0.50", 0.5), ("validation_f1", threshold)]):
    predictions = (test_score >= cutoff).astype(int)
    reports[name] = classification_report(y_test, predictions, labels=[0, 1], output_dict=True, zero_division=0)
    for row, normalize in enumerate([None, "true"]):
        ConfusionMatrixDisplay.from_predictions(
            y_test, predictions, labels=[0, 1], normalize=normalize,
            values_format="d" if normalize is None else ".1%", ax=axes[row, col],
            cmap="Blues", colorbar=False,
        )
        axes[row, col].set_title(f"Ngưỡng {cutoff:.3f} · " + ("Số mẫu" if row == 0 else "Tỷ lệ theo lớp thật"))
finish_plot(fig, "06_test_confusion_matrices.png")

per_class = pd.concat({key: pd.DataFrame(report).T.loc[["0", "1"]] for key, report in reports.items()})
display(per_class.round(4))
per_class.to_csv(OUTPUT_DIR / "brfss_test_per_class.csv")
fig, ax = plt.subplots(figsize=(10, 4))
labels = [f"{name} / lớp {label}" for name, label in per_class.index]
per_class[["precision", "recall", "f1-score"]].set_axis(labels).plot.bar(ax=ax, rot=15)
ax.set(ylim=(0, 1.05), title="Precision, Recall và F1 theo từng lớp trên test")
finish_plot(fig, "07_test_per_class.png")

In [ ]:
# Đóng gói bộ chuẩn hóa và mạng đã chọn.
pipeline = Pipeline([("scale", scaler), ("mlp", best_model)])
prediction = (test_score >= threshold).astype(int)
bundle = {
    "pipeline": pipeline, "features": FEATURES, "threshold": threshold,
    "metrics": {
        "average_precision": ap, "roc_auc": auc,
        "accuracy": float(accuracy_score(y_test, prediction)),
        "precision": reports["validation_f1"]["1"]["precision"],
        "recall": reports["validation_f1"]["1"]["recall"],
        "f1": reports["validation_f1"]["1"]["f1-score"],
    },
    "dataset": DATA_FILE, "target": TARGET, "model_version": "brfss_web5_mlp_v1",
    "sklearn_version": sklearn.__version__,
}
destination = OUTPUT_DIR / "brfss_web5.bin"
joblib.dump(bundle, destination, compress=3)
reloaded = joblib.load(destination)
np.testing.assert_allclose(reloaded["pipeline"].predict_proba(x.iloc[test[:8]])[:, 1], test_score[:8], atol=1e-6)
print(f"Saved and verified: {destination}")

## Mức đóng góp của từng thuộc tính

Đo **Permutation Importance** trên validation giữ phân bố nhãn gốc. Với mỗi thuộc tính, xáo trộn các giá trị của riêng cột đó 10 lần rồi đo **AP gốc − AP sau khi xáo trộn**. Cột dài hơn nghĩa là mô hình bản 5 thuộc tính dựa vào thông tin đó nhiều hơn. Thanh sai số là độ lệch chuẩn giữa các lần xáo trộn. Kết quả phụ thuộc vào tập biến và mô hình, vì các biến tương quan có thể mang thông tin trùng nhau.

Lý do chọn 5 trường còn dựa trên thí nghiệm đối chiếu trước đó trên cùng tập test BRFSS. AP lần lượt là 0,4169 cho bản 5 trường, 0,4150 cho bản 8 trường đang dùng và 0,4227 cho bản đủ 21 trường. Các số này đến từ lần chạy đối chiếu với seed 42. Biểu đồ bên dưới giải thích mức đóng góp trong bản 5 trường, còn bảng điểm của các cấu hình cho thấy mức đánh đổi khi rút gọn form.

In [ ]:
with threadpool_limits(limits=2, user_api="blas"):
    permuted = permutation_importance(
        pipeline, x.iloc[val], y_val, scoring="average_precision",
        n_repeats=IMPORTANCE_REPEATS, random_state=SEED, n_jobs=1,
    )

importance = pd.DataFrame({
    "feature": FEATURES,
    "mean_ap_drop": permuted.importances_mean,
    "std_ap_drop": permuted.importances_std,
}).sort_values("mean_ap_drop", ascending=False)
importance.to_csv(OUTPUT_DIR / "brfss_web5_feature_importance.csv", index=False)
display(importance.round(5))

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(importance["feature"], importance["mean_ap_drop"],
        xerr=importance["std_ap_drop"], color="#1764ac", capsize=4)
ax.invert_yaxis()
ax.axvline(0, color="gray", linewidth=1)
ax.set(
    xlabel="Mức giảm Average Precision (AP)",
    title=f"Đóng góp của 5 thuộc tính trên validation (AP gốc {best_ap:.4f})",
)
finish_plot(fig, "08_web5_permutation_importance.png")

## Dùng model

Notebook xuất `/kaggle/working/brfss_web5.bin`, các biểu đồ PNG và bảng CSV. File .bin dùng joblib và chứa pipeline chuẩn hóa + MLP, thứ tự cột, ngưỡng và phiên bản scikit-learn. Dùng phiên bản scikit-learn trong artifact khi nạp lại.

`GenHlth` mã hóa từ 1 (xuất sắc) đến 5 (kém). `BMI` tính bằng kg/m². `HighBP` và `HighChol` ghi nhận từng được nhân viên y tế báo chỉ số cao (1 = Có, 0 = Không). `Age` dùng mã nhóm BRFSS từ 1 (18–24 tuổi) đến 13 (từ 80 tuổi). Đầu ra là điểm phân loại tình trạng được báo cáo trong khảo sát.